# Notebook 04b — Robustness Checks

An independent code review raised four questions about the main results. Each check below changes **one thing**,
re-tunes LightGBM from scratch (same search as Notebook 03), and compares the result with the main feature set B
LightGBM model on the **same test respondents** using a paired DeLong test.

| Check | What changes | The question it answers |
|---|---|---|
| R1 Strict B | Remove 4 more features that may be **consequences** of stroke: vision difficulty, insurance type, depressive disorder, poor mental-health days | Do the results depend on hidden post-stroke clues? |
| R2 No missing flags | No "was missing" columns; blanks filled with the median / most common answer only | Do the results depend on *who skipped* questions? |
| R3 Strict B + no flags | R1 and R2 together | The most conservative model |
| R4 All respondents | Train **and** test on everyone, including partial interviews | Does the partial-interview filter change anything? |
| R5 Selected features, own tuning | The 18 selected features, with hyperparameters tuned **for them** | Was the feature-selection penalty in Notebook 03b caused by borrowed tuning? |

Nothing chosen here feeds back into the main model; the test set is only used to report.

**Input** — `brfss_model_ready_v2.csv`, `feature_config_v2.json`, `v2_selected_features.json`, `models/v2/primary_B_LightGBM.joblib`
**Output** — `outputs/v2_robustness.csv`, `models/v2/robustness/*.joblib`

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])

import pandas as pd
import numpy as np
import os, json, time, joblib
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import rankdata, norm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/stroke_prediction'
READY_PATH  = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_model_ready_v2.csv')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
MODELS_DIR  = os.path.join(BASE_DIR, 'models', 'v2')
ROB_DIR     = os.path.join(MODELS_DIR, 'robustness')
os.makedirs(ROB_DIR, exist_ok=True)

# ── Run settings (same search as Notebook 03) ─────────────────────────────
TUNE_N   = 100_000
CV_FOLDS = 3
N_ITER   = 15
SEED     = 42
STRICT_REMOVE = ['BLIND', 'PRIMINS1', 'ADDEPEV3', 'MENTHLTH']   # possible consequences of stroke

config   = json.load(open(os.path.join(OUTPUTS_DIR, 'feature_config_v2.json')))
selected = json.load(open(os.path.join(OUTPUTS_DIR, 'v2_selected_features.json')))['features_S']
FEATURES_B = config['features_B']
STRICT_B   = [f for f in FEATURES_B if f not in STRICT_REMOVE]

df_all = pd.read_csv(READY_PATH)
primary = df_all['meta_keep_primary'].to_numpy(bool)
train_p = df_all[(df_all['split'] == 'train') & primary]
train_a = df_all[df_all['split'] == 'train']
val_p   = df_all[(df_all['split'] == 'validation') & primary]
test_p  = df_all[(df_all['split'] == 'test') & primary].reset_index(drop=True)
test_a  = df_all[df_all['split'] == 'test'].reset_index(drop=True)
print(f"train primary {len(train_p):,} | train all {len(train_a):,} | test primary {len(test_p):,} | test all {len(test_a):,}")
print(f"Strict B: {len(STRICT_B)} features (removed {', '.join(STRICT_REMOVE)}) | selected: {len(selected)}")

### The pipeline, with "was missing" flags switchable
With `flags=False`, numeric and yes/no blanks are filled with the training median / most common answer **without**
a flag column, and blanks in categorical questions (e.g. marital status) get the most common category instead of
their own "missing" category — so the model cannot see that a question was skipped.

In [ ]:
def make_preprocessor(features, flags=True):
    num  = [f for f in config['continuous'] + config['ordinal'] if f in features]
    binr = [f for f in config['binary'] if f in features]
    nom  = [f for f in config['nominal'] if f in features]
    nom_fill = SimpleImputer(strategy='constant', fill_value=-1) if flags else SimpleImputer(strategy='most_frequent')
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='median', add_indicator=flags), num),
        ('bin', SimpleImputer(strategy='most_frequent', add_indicator=flags), binr),
        ('nom', Pipeline([('impute', nom_fill),
                          ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), nom),
    ])

def make_lgbm():
    return LGBMClassifier(class_weight='balanced', subsample_freq=1, n_jobs=-1, random_state=SEED, verbose=-1)

GRID = {'model__n_estimators': [200, 400, 600], 'model__num_leaves': [15, 31, 63],
        'model__learning_rate': [0.03, 0.05, 0.1], 'model__min_child_samples': [20, 50, 100],
        'model__subsample': [0.7, 0.9], 'model__colsample_bytree': [0.7, 0.9]}

def delong(y, preds):
    preds = np.atleast_2d(preds)
    pos, neg = preds[:, y == 1], preds[:, y == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([rankdata(r) for r in pos])
    ty = np.array([rankdata(r) for r in neg])
    tz = np.array([rankdata(r) for r in np.hstack([pos, neg])])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1) / (2 * n)
    cov = np.atleast_2d(np.cov((tz[:, :m] - tx) / n)) / m + np.atleast_2d(np.cov(1 - (tz[:, m:] - ty) / m)) / n
    return aucs, cov

### Fit every variant (resumable — finished models load from Drive)

In [ ]:
VARIANTS = {
    'R1 Strict B'                    : dict(features=STRICT_B,   flags=True,  train=train_p),
    'R2 No missing flags'            : dict(features=FEATURES_B, flags=False, train=train_p),
    'R3 Strict B + no flags'         : dict(features=STRICT_B,   flags=False, train=train_p),
    'R4 All respondents'             : dict(features=FEATURES_B, flags=True,  train=train_a),
    'R5 Selected features, own tuning': dict(features=selected,  flags=True,  train=train_p),
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
fitted, params = {}, {}
for name, v in VARIANTS.items():
    path = os.path.join(ROB_DIR, name.split()[0] + '.joblib')
    if os.path.exists(path):
        fitted[name] = joblib.load(path)
        print(f"{name:<34} loaded from disk")
        continue
    start = time.time()
    X, y = v['train'][v['features']], v['train']['stroke']
    X_tune, _, y_tune, _ = train_test_split(X, y, train_size=min(TUNE_N, len(X) - 1), stratify=y, random_state=SEED)
    pipe = Pipeline([('prep', make_preprocessor(v['features'], v['flags'])), ('model', make_lgbm())])
    search = RandomizedSearchCV(pipe, GRID, n_iter=N_ITER, cv=cv, scoring='roc_auc', random_state=SEED, n_jobs=1)
    search.fit(X_tune, y_tune)
    fitted[name] = clone(pipe).set_params(**search.best_params_).fit(X, y)
    joblib.dump(fitted[name], path)
    print(f"{name:<34} CV AUC {search.best_score_:.4f} | {(time.time()-start)/60:.1f} min")

reference = joblib.load(os.path.join(MODELS_DIR, 'primary_B_LightGBM.joblib'))

### Results — each variant vs the main feature set B LightGBM

* **Δ AUC** = variant − reference on the **same** people, with a 95% CI and a paired DeLong p-value.
* R4 is compared on the primary test set (the same people as everywhere else) and also reported on **all** test
  respondents.
* How to read it: a Δ AUC close to 0 means the main result **does not depend** on what that check removed.

In [ ]:
y_test = test_p['stroke'].to_numpy()
p_ref = reference.predict_proba(test_p[FEATURES_B])[:, 1]
auc_ref, cov_ref = delong(y_test, p_ref)
rows = [{'variant': 'Reference: B LightGBM (Notebook 03)', 'n_features': len(FEATURES_B),
         'val_auc': roc_auc_score(val_p['stroke'], reference.predict_proba(val_p[FEATURES_B])[:, 1]),
         'test_auc': auc_ref[0], 'test_auc_low': auc_ref[0] - 1.96 * np.sqrt(cov_ref[0, 0]),
         'test_auc_high': auc_ref[0] + 1.96 * np.sqrt(cov_ref[0, 0]),
         'test_ap': average_precision_score(y_test, p_ref)}]

for name, v in VARIANTS.items():
    m, feats = fitted[name], v['features']
    p = m.predict_proba(test_p[feats])[:, 1]
    aucs, cov = delong(y_test, np.vstack([p, p_ref]))
    diff = aucs[0] - aucs[1]
    se = np.sqrt(max(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1], 1e-12))
    row = {'variant': name, 'n_features': len(feats),
           'val_auc': roc_auc_score(val_p['stroke'], m.predict_proba(val_p[feats])[:, 1]),
           'test_auc': aucs[0], 'test_auc_low': aucs[0] - 1.96 * np.sqrt(cov[0, 0]),
           'test_auc_high': aucs[0] + 1.96 * np.sqrt(cov[0, 0]), 'test_ap': average_precision_score(y_test, p),
           'delta_auc': diff, 'delta_low': diff - 1.96 * se, 'delta_high': diff + 1.96 * se,
           'p_delong': 2 * norm.sf(abs(diff / se))}
    if name.startswith('R4'):
        row['test_auc_all_respondents'] = roc_auc_score(test_a['stroke'], m.predict_proba(test_a[feats])[:, 1])
    rows.append(row)

rob = pd.DataFrame(rows)
rob.round(4).to_csv(os.path.join(OUTPUTS_DIR, 'v2_robustness.csv'), index=False)

show = pd.DataFrame({
    'features': rob['n_features'].values,
    'val AUC': rob['val_auc'].map('{:.4f}'.format).values,
    'test AUC (95% CI)': [f"{a:.4f} ({l:.4f}–{h:.4f})" for a, l, h in zip(rob['test_auc'], rob['test_auc_low'], rob['test_auc_high'])],
    'Δ AUC vs reference (95% CI)': ['—'] + [f"{d:+.4f} ({l:+.4f} to {h:+.4f})" for d, l, h in
                                            zip(rob['delta_auc'][1:], rob['delta_low'][1:], rob['delta_high'][1:])],
    'p': ['—'] + [f"{p:.3g}" for p in rob['p_delong'][1:]],
}, index=rob['variant'].values)
print(show.to_string())
r4 = rob[rob['variant'].str.startswith('R4')].iloc[0]
print(f"\nR4 on ALL test respondents (incl. partial interviews): AUC {r4['test_auc_all_respondents']:.4f}")
print("\n" + "=" * 60 + "\n  Notebook 04b complete -> outputs/v2_robustness.csv\n" + "=" * 60)